In [2]:
# Importing libraries

import pandas as pd
import numpy as np
from pathlib import Path

In [3]:
# Setting paths

DATA_PATH = Path("../data/processed/london_air_quality_weather_2021_2024.csv")
OUTPUT_PATH = Path("../data/processed/london_air_quality_features_2021_2024.csv")
FIGURE_PATH = Path("../outputs/figures/random_forest_feature_importance.png")

FIGURE_PATH.parent.mkdir(parents=True, exist_ok=True)

In [4]:
# Loading data

df = pd.read_csv(DATA_PATH)
df["Datetime"] = pd.to_datetime(df["Datetime"])

print("Shape:", df.shape)
print("Date range:", df["Datetime"].min(), "→", df["Datetime"].max())

Shape: (168015, 13)
Date range: 2021-01-01 01:00:00 → 2024-12-31 23:00:00


In [5]:
# Checking data types

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 168015 entries, 0 to 168014
Data columns (total 13 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   Datetime       168015 non-null  datetime64[ns]
 1   Station        168015 non-null  object        
 2   NO2            130270 non-null  float64       
 3   PM10           162210 non-null  float64       
 4   PM2.5          154284 non-null  float64       
 5   O3             132693 non-null  float64       
 6   Temperature    167975 non-null  float64       
 7   Dewpoint       167975 non-null  float64       
 8   Humidity       167960 non-null  float64       
 9   WindSpeed      165205 non-null  float64       
 10  WindDirection  165205 non-null  float64       
 11  Pressure       167965 non-null  float64       
 12  Visibility     167085 non-null  float64       
dtypes: datetime64[ns](1), float64(11), object(1)
memory usage: 16.7+ MB


In [6]:
# Sorting data

df = (
    df.sort_values(["Station", "Datetime"])
      .drop_duplicates(["Station", "Datetime"])
      .reset_index(drop=True)
)

print("Shape:", df.shape)
print("Duplicate Station-Datetime rows:", df.duplicated(["Station", "Datetime"]).sum())

Shape: (168015, 13)
Duplicate Station-Datetime rows: 0


In [7]:
# Creating time features

df["Year"] = df["Datetime"].dt.year
df["Month"] = df["Datetime"].dt.month
df["Day"] = df["Datetime"].dt.day
df["Hour"] = df["Datetime"].dt.hour
df["DayOfWeek"] = df["Datetime"].dt.dayofweek
df["DayOfYear"] = df["Datetime"].dt.dayofyear
df["WeekOfYear"] = df["Datetime"].dt.isocalendar().week.astype(int)

In [8]:
# Creating weekend feature

df["IsWeekend"] = (df["DayOfWeek"] >= 5).astype(int)

In [9]:
# Creating seasonal feature

df["Season"] = df["Month"].map({
    12: "Winter", 1: "Winter", 2: "Winter",
    3: "Spring", 4: "Spring", 5: "Spring",
    6: "Summer", 7: "Summer", 8: "Summer",
    9: "Autumn", 10: "Autumn", 11: "Autumn"
})

### Pollution history

Previous PM2.5 values are used as predictive features. Lags are matched by exact timestamps so gaps are not treated as consecutive hours.

In [10]:
# Creating exact PM2.5 lag features

pm25_lookup = df[["Station", "Datetime", "PM2.5"]].copy()

for hours in [1, 3, 6, 24]:
    lag_lookup = pm25_lookup.rename(
        columns={
            "Datetime": "Lag_Datetime",
            "PM2.5": f"PM2.5_lag_{hours}h"
        }
    )

    df["Lag_Datetime"] = df["Datetime"] - pd.Timedelta(hours=hours)

    df = df.merge(
        lag_lookup,
        left_on=["Station", "Lag_Datetime"],
        right_on=["Station", "Lag_Datetime"],
        how="left",
        validate="one_to_one"
    ).drop(columns=["Lag_Datetime"])

In [11]:
# Creating rolling PM2.5 features

rolling = (
    df[["Station", "Datetime", "PM2.5"]]
    .sort_values(["Station", "Datetime"])
    .set_index("Datetime")
    .groupby("Station")["PM2.5"]
    .rolling("3h", closed="left")
    .mean()
    .rename("PM2.5_rolling_3h")
    .reset_index()
)

rolling6 = (
    df[["Station", "Datetime", "PM2.5"]]
    .sort_values(["Station", "Datetime"])
    .set_index("Datetime")
    .groupby("Station")["PM2.5"]
    .rolling("6h", closed="left")
    .mean()
    .rename("PM2.5_rolling_6h")
    .reset_index()
)

rolling24 = (
    df[["Station", "Datetime", "PM2.5"]]
    .sort_values(["Station", "Datetime"])
    .set_index("Datetime")
    .groupby("Station")["PM2.5"]
    .rolling("24h", closed="left")
    .mean()
    .rename("PM2.5_rolling_24h")
    .reset_index()
)

for roll_df in [rolling, rolling6, rolling24]:
    df = df.merge(
        roll_df,
        on=["Station", "Datetime"],
        how="left",
        validate="one_to_one"
    )

In [12]:
# Creating weather features

df["Temperature_Humidity"] = (
    df["Temperature"] * df["Humidity"]
)

df["WindSpeed_Squared"] = (
    df["WindSpeed"] ** 2
)

df["Pressure_Change"] = (
    df.groupby("Station")["Pressure"].diff()
)

In [13]:
# Checking engineered features

feature_columns = [
    "Year", "Month", "Day", "Hour", "DayOfWeek", "DayOfYear",
    "WeekOfYear", "IsWeekend", "Season",
    "PM2.5_lag_1h", "PM2.5_lag_3h", "PM2.5_lag_6h",
    "PM2.5_lag_24h", "PM2.5_rolling_3h", "PM2.5_rolling_6h",
    "PM2.5_rolling_24h", "Temperature_Humidity",
    "WindSpeed_Squared", "Pressure_Change"
]

print("Number of engineered features:", len(feature_columns))

Number of engineered features: 19


In [14]:
# Checking stations

print(df["Station"].unique())
print()
print(df["Station"].value_counts())

['London Bloomsbury' 'London Harlington' 'London Honor Oak Park'
 'London Marylebone Road' 'London N. Kensington']

Station
London Bloomsbury         33603
London Harlington         33603
London Honor Oak Park     33603
London Marylebone Road    33603
London N. Kensington      33603
Name: count, dtype: int64


In [15]:
# Checking engineered missing values

missing_features = (
    df[feature_columns]
    .isna()
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
)

print(missing_features[missing_features > 0])

PM2.5_lag_3h            12.19
PM2.5_lag_6h            12.19
PM2.5_lag_1h            12.18
PM2.5_lag_24h            8.24
PM2.5_rolling_3h         7.88
PM2.5_rolling_6h         7.74
PM2.5_rolling_24h        7.23
WindSpeed_Squared        1.67
Pressure_Change          0.06
Temperature_Humidity     0.04
dtype: float64


In [16]:
# Creating exact next-hour PM2.5 target

target_lookup = (
    df[["Station", "Datetime", "PM2.5"]]
    .rename(columns={"PM2.5": "PM2.5_target"})
)

df["Target_Datetime"] = (
    df["Datetime"] + pd.Timedelta(hours=1)
)

df = df.merge(
    target_lookup.rename(columns={"Datetime": "Target_Datetime"}),
    on=["Station", "Target_Datetime"],
    how="left",
    validate="one_to_one"
)

df = df.drop(columns=["Target_Datetime"])

print("Missing PM2.5 targets:", df["PM2.5_target"].isna().sum())

Missing PM2.5 targets: 20459


In [17]:
# Validating target alignment

validation = df.loc[
    df["PM2.5_target"].notna(),
    ["Station", "Datetime", "PM2.5_target"]
].copy()

expected = (
    df[["Station", "Datetime", "PM2.5"]]
    .rename(columns={
        "Datetime": "Target_Datetime",
        "PM2.5": "Expected_target"
    })
)

validation["Target_Datetime"] = (
    validation["Datetime"] + pd.Timedelta(hours=1)
)

validation = validation.merge(
    expected,
    on=["Station", "Target_Datetime"],
    how="left",
    validate="one_to_one"
)

alignment_errors = (
    validation["PM2.5_target"] != validation["Expected_target"]
).sum()

print("Alignment errors:", alignment_errors)


Alignment errors: 0


In [18]:
# Checking target creation

print("Shape:", df.shape)
print("Duplicate Station-Datetime rows:", df.duplicated(["Station", "Datetime"]).sum())
print("Missing targets:", df["PM2.5_target"].isna().sum())

Shape: (168015, 33)
Duplicate Station-Datetime rows: 0
Missing targets: 20459


In [19]:
# Removing rows without a target

df = df.dropna(subset=["PM2.5_target"]).copy()

print("Shape:", df.shape)
print("Missing targets:", df["PM2.5_target"].isna().sum())

Shape: (147556, 33)
Missing targets: 0


In [20]:
# Removing rows without current PM2.5

df = df.dropna(subset=["PM2.5"]).copy()

print("Shape:", df.shape)
print("Missing PM2.5:", df["PM2.5"].isna().sum())

Shape: (147228, 33)
Missing PM2.5: 0


In [21]:
# Final dataset check

print("Shape:", df.shape)
print("Date range:", df["Datetime"].min(), "→", df["Datetime"].max())
print("Stations:", df["Station"].nunique())
print("Duplicate Station-Datetime rows:", df.duplicated(["Station", "Datetime"]).sum())

Shape: (147228, 33)
Date range: 2021-01-01 01:00:00 → 2024-12-31 22:00:00
Stations: 5
Duplicate Station-Datetime rows: 0


In [22]:
# Creating time-based train/test split

train_df = df[df["Datetime"].dt.year <= 2023].copy()
test_df = df[df["Datetime"].dt.year == 2024].copy()

print("Training shape:", train_df.shape)
print("Training period:", train_df["Datetime"].min(), "→", train_df["Datetime"].max())

print()
print("Testing shape:", test_df.shape)
print("Testing period:", test_df["Datetime"].min(), "→", test_df["Datetime"].max())

Training shape: (108102, 33)
Training period: 2021-01-01 01:00:00 → 2023-12-31 22:00:00

Testing shape: (39126, 33)
Testing period: 2024-01-01 01:00:00 → 2024-12-31 22:00:00


In [23]:
# Defining model features

target = "PM2.5_target"

features = [
    "NO2", "PM10", "PM2.5", "O3",
    "Temperature", "Dewpoint", "Humidity",
    "WindSpeed", "WindDirection", "Pressure", "Visibility",
    "Year", "Month", "Day", "Hour", "DayOfWeek",
    "DayOfYear", "WeekOfYear", "IsWeekend",
    "PM2.5_lag_1h", "PM2.5_lag_3h", "PM2.5_lag_6h",
    "PM2.5_lag_24h", "PM2.5_rolling_3h",
    "PM2.5_rolling_6h", "PM2.5_rolling_24h",
    "Temperature_Humidity", "WindSpeed_Squared",
    "Pressure_Change"
]

print("Number of features:", len(features))
print("Target:", target)

Number of features: 29
Target: PM2.5_target


In [24]:
# Creating model datasets

X_train = train_df[features].copy()
X_test = test_df[features].copy()

y_train = train_df[target].copy()
y_test = test_df[target].copy()

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (108102, 29)
X_test: (39126, 29)
y_train: (108102,)
y_test: (39126,)


In [25]:
# Importing imputer

from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

X_train_imputed = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_test_imputed = pd.DataFrame(
    imputer.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

print("Training missing values:", X_train_imputed.isna().sum().sum())
print("Testing missing values:", X_test_imputed.isna().sum().sum())

Training missing values: 0
Testing missing values: 0


In [26]:
# Importing scaler

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

print("X_train_scaled:", X_train_scaled.shape)
print("X_test_scaled:", X_test_scaled.shape)

X_train_scaled: (108102, 29)
X_test_scaled: (39126, 29)


In [27]:
# Checking target values

print("Training target:")
print(y_train.describe())

print()
print("Testing target:")
print(y_test.describe())

print()
print("Training missing targets:", y_train.isna().sum())
print("Testing missing targets:", y_test.isna().sum())

Training target:
count    108102.000000
mean          8.594547
std           7.102222
min           0.000000
25%           4.245000
50%           6.604000
75%          10.472000
max          96.793000
Name: PM2.5_target, dtype: float64

Testing target:
count    39126.000000
mean         7.595094
std          5.991856
min          0.000000
25%          3.797000
50%          5.896000
75%          9.360750
max        121.321000
Name: PM2.5_target, dtype: float64

Training missing targets: 0
Testing missing targets: 0


In [28]:
# Saving engineered dataset

df.to_csv(OUTPUT_PATH, index=False)

print("Saved:", OUTPUT_PATH)
print("Shape:", df.shape)

Saved: ..\data\processed\london_air_quality_features_2021_2024.csv
Shape: (147228, 33)


In [29]:
# Final quality control

print("=" * 60)
print("FINAL FEATURE DATASET CHECK")
print("=" * 60)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))
print("Stations:", df["Station"].nunique())
print("Missing targets:", df["PM2.5_target"].isna().sum())
print("Duplicate Station-Datetime rows:", df.duplicated(["Station", "Datetime"]).sum())
print("Saved:", OUTPUT_PATH)

FINAL FEATURE DATASET CHECK
Rows: 147,228
Columns: 33
Stations: 5
Missing targets: 0
Duplicate Station-Datetime rows: 0
Saved: ..\data\processed\london_air_quality_features_2021_2024.csv


In [2]:
# Identify the dataset objects currently available

print("=" * 60)
print("DATASET OBJECT AUDIT")
print("=" * 60)

for name in [
    "df",
    "merged_df",
    "feature_df",
    "model_df",
    "final_df",
    "train_df",
    "test_df",
    "X_train",
    "X_test",
    "y_train",
    "y_test"
]:
    if name in globals():
        obj = globals()[name]

        if hasattr(obj, "shape"):
            print(f"{name}: {obj.shape}")
        else:
            print(f"{name}: exists")

DATASET OBJECT AUDIT
